<a href="https://colab.research.google.com/github/vannoordenne/static./blob/main/webcam2interrogator2images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This Colab contains the main code that was used for the interactive art installation as a case study for the MSc thesis "IIA4XAI: Interactive Installation Art to increase Non-Expert User Engagement with AI Systems" by Marise van Noordenne.

This program is a combined adaptation of:

Image 2 prompt generation by CLIP Interrogator: 
https://colab.research.google.com/github/pharmapsychotic/clip-interrogator/blob/main/clip_interrogator.ipynb

Prompt 2 image generation by Stable Diffusion: 
https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/stable_diffusion.ipynb. 

Displaying the generation process by Edan Myer: 
https://colab.research.google.com/drive/1_kbRZPTjnFgViPrmGcUsaszEdYa8XTpq?usp=sharing

In [ ]:
#@title setup
!nvidia-smi
!pip install diffusers
!pip install transformers scipy ftfy accelerate
!pip install "ipywidgets>=7,<8"
!pip install gradio
!pip install open_clip_torch
!pip install clip-interrogator

from PIL import Image as Img
from IPython.display import display, Javascript, Image, HTML, clear_output
from google.colab.output import eval_js
from base64 import b64decode, b64encode

import csv
import subprocess
import os
import cv2
import numpy as np
import torch
torch_device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()

import gradio as gr
from clip_interrogator import Config, Interrogator

from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel, PNDMScheduler, LMSDiscreteScheduler
from diffusers.schedulers.scheduling_ddim import DDIMScheduler
from tqdm import tqdm
from tqdm.auto import tqdm
from torch import autocast
from torch.nn import functional as F
from PIL import Image, ImageDraw
from huggingface_hub import notebook_login
from pathlib import Path

# --- Configuration (cleaned for public release; see .env.example and SECURITY.md) ---

def get_config(name: str, default: str | None = None) -> str | None:
    """Read config from env, Colab Secrets, or default (in that order)."""
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return default

# Hugging Face auth: .env locally, Colab Secrets (key: HF_TOKEN), or interactive login.
HF_TOKEN = get_config("HF_TOKEN")
if not HF_TOKEN:
    notebook_login()  # fallback if no secret is set
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# Output paths — optional Colab Secrets: IIA4XAI_OUTPUT_DIR, IIA4XAI_PROMPT_FILE
OUTPUT_DIR = Path(get_config("IIA4XAI_OUTPUT_DIR", "./output/imgs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROMPT_FILE = Path(get_config("IIA4XAI_PROMPT_FILE", "./output/prompt.txt"))
PROMPT_FILE.parent.mkdir(parents=True, exist_ok=True)

os.chdir(OUTPUT_DIR)
file_path = OUTPUT_DIR / "image.png"

caption_model_name = 'blip-large' #@param ["blip-base", "blip-large", "git-large-coco"]
clip_model_name = 'ViT-L-14/openai' #@param ["ViT-L-14/openai", "ViT-H-14/laion2b_s32b_b79k"]

config = Config()
config.clip_model_name = clip_model_name
config.caption_model_name = caption_model_name
ci = Interrogator(config)

# 1. Load the autoencoder model which will be used to decode the latents into image space.
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae", token=HF_TOKEN)
vae = vae.to(torch_device)

# 2. Load the tokenizer and text encoder to tokenize and encode the text. 
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = text_encoder.to(torch_device)

# 3. The UNet model for generating the latents.
unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet", token=HF_TOKEN)
unet = unet.to(torch_device) 

scheduler = DDIMScheduler(
    beta_start=0.00085, beta_end=0.012,
    beta_schedule='scaled_linear', num_train_timesteps=1000)

In [ ]:
#@title prompt 2 img

def get_text_embeds(prompt):
  # Tokenize text and get embeddings
  text_input = tokenizer(
      prompt, padding='max_length', max_length=tokenizer.model_max_length,
      truncation=True, return_tensors='pt')
  with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]

  # Do the same for unconditional embeddings
  uncond_input = tokenizer(
      [''] * len(prompt), padding='max_length',
      max_length=tokenizer.model_max_length, return_tensors='pt')
  with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]

  # Cat for final embeddings
  text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
  return text_embeddings

# test_embeds = get_text_embeds(['portrait photo, Canon 60D, 50mm'])
# print(test_embeds)
# print(test_embeds.shape)


def produce_latents(text_embeddings, height=512, width=512,
                    num_inference_steps=50, guidance_scale=7.5, latents=None,
                    return_all_latents=False, start_step=10):
  if latents is None:
    latents = torch.randn((text_embeddings.shape[0] // 2, unet.in_channels, \
                           height // 8, width // 8))
  latents = latents.to(torch_device)

  scheduler.set_timesteps(num_inference_steps)
  # latents = latents * scheduler.sigmas[0]

  if start_step > 0:
    start_timestep = scheduler.timesteps[start_step]
    start_timesteps = start_timestep.repeat(latents.shape[0]).long()

    noise = torch.randn_like(latents)
    latents = scheduler.add_noise(latents, noise, start_timesteps)

  latent_list = [latents]

  with autocast('cuda'):
    for i, t in tqdm(enumerate(scheduler.timesteps[start_step:])):
      # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
      latent_model_input = torch.cat([latents] * 2)
      # sigma = scheduler.sigmas[i]
      # latent_model_input = latent_model_input / ((sigma**2 + 1) ** 0.5)

      # predict the noise residual
      with torch.no_grad():
        noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings)['sample']

      # perform guidance
      noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
      noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

      # compute the previous noisy sample x_t -> x_t-1
      latents = scheduler.step(noise_pred, t, latents)['prev_sample']
      latent_list.append(latents)

    if not return_all_latents:
      return latents

    all_latents = torch.cat(latent_list, dim=0)
    return all_latents


# test_latents = produce_latents(test_embeds)
# print(test_latents)
# print(test_latents.shape)


def decode_img_latents(latents):
  latents = 1 / 0.18215 * latents

  with torch.no_grad():
    imgs = vae.decode(latents)

  imgs = (imgs.sample / 2 + 0.5).clamp(0, 1)
  imgs = imgs.detach().cpu().permute(0, 2, 3, 1).numpy()
  imgs = (imgs * 255).round().astype('uint8')
  pil_images = [Image.fromarray(image) for image in imgs]
  return pil_images

# imgs = decode_img_latents(test_latents)
# imgs[0]

def encode_img_latents(imgs):
  if not isinstance(imgs, list):
    imgs = [imgs]

  img_arr = np.stack([np.array(img) for img in imgs], axis=0)
  img_arr = img_arr / 255.0
  img_arr = torch.from_numpy(img_arr).float().permute(0, 3, 1, 2)
  img_arr = 2 * (img_arr - 0.5)

  latent_dists = vae.encode(img_arr.to(torch_device))[0]
  latent_samples = latent_dists.sample()
  latent_samples *= 0.18215

  return latent_samples

def prompt_to_img(prompts, height, width, num_inference_steps=50,
                  guidance_scale=7.5, latents=None, return_all_latents=False, batch_size=1, start_step=0, folder=0):
  if isinstance(prompts, str):
    prompts = [prompts]

  # Prompts -> text embeds
  text_embeds = get_text_embeds(prompts)

  # Text embeds -> img latents
  latents = produce_latents(
      text_embeds, height=height, width=width, latents=latents,
      num_inference_steps=num_inference_steps, guidance_scale=guidance_scale, return_all_latents=return_all_latents, start_step=start_step)
  
  # Img latents -> imgs
  all_imgs = []
  for i in tqdm(range(0, len(latents), batch_size)):
    imgs = decode_img_latents(latents[i:i+batch_size])
    all_imgs.extend(imgs)
    # Save each frame to OUTPUT_DIR/<folder>/ (see setup cell for path configuration)
    save_dir = OUTPUT_DIR / str(folder)
    save_dir.mkdir(parents=True, exist_ok=True)
    all_imgs[i].save(save_dir / f"image{i}.png")

  return all_imgs

def image_to_prompt(image, mode):
    ci.config.chunk_size = 2048 if ci.config.clip_model_name == "ViT-L-14/openai" else 1024
    ci.config.flavor_intermediate_count = 2048 if ci.config.clip_model_name == "ViT-L-14/openai" else 1024
    image = image.convert('RGB')
    if mode == 'best':
        return ci.interrogate(image)
    elif mode == 'classic':
        return ci.interrogate_classic(image)
    elif mode == 'fast':
        return ci.interrogate_fast(image)
    elif mode == 'negative':
        return ci.interrogate_negative(image)

In [ ]:
#@title capture webcam (live — run before webcam 2 prompt)
# Original Colab capture from the thesis prototype. Click Capture after allowing camera access.

WEBCAM_FILENAME = "webcam.png"
webcam_path = OUTPUT_DIR / WEBCAM_FILENAME
input_source = "webcam" #@param ["webcam", "upload"]
capture_quality = 1.0 #@param {type:"number"}
apply_install_crop = True #@param {type:"boolean"}  # thesis installation framing (from static v1)
crop_y_end = 480 #@param {type:"integer"}
crop_x_start = 140 #@param {type:"integer"}
crop_x_end = 500 #@param {type:"integer"}
output_size = 512 #@param {type:"integer"}


def postprocess_install_framing(path: Path) -> None:
    """Crop and resize capture to match the original static v1 installation framing."""
    if not apply_install_crop:
        return
    image = cv2.imread(str(path))
    if image is None:
        print(f"Could not read {path} for cropping.")
        return
    cropped = image[0:crop_y_end, crop_x_start:crop_x_end]
    resized = cv2.resize(cropped, (output_size, output_size))
    cv2.imwrite(str(path), resized)
    print(f"Applied install crop → {path} ({output_size}x{output_size})")


def capture_webcam_to_file(dest: Path, quality: float = 1.0) -> None:
    """Live browser webcam capture — uses Colab iframe resize (required for camera access)."""
    js = Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = 'Capture';
        div.appendChild(capture);

        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});

        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // Required in Colab so the browser grants camera access in the notebook iframe.
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        await new Promise((resolve) => capture.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        video.remove();
        return canvas.toDataURL('image/png', quality);
      }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    dest.write_bytes(b64decode(data.split(',')[1]))
    postprocess_install_framing(dest)
    print(f'Saved to {dest}')
    display(Image(str(dest)))


if input_source == "webcam":
    try:
        capture_webcam_to_file(webcam_path, quality=capture_quality)
    except Exception as err:
        print(f"Webcam capture failed: {err}")
        print("Fallback: set input_source to 'upload' and re-run this cell.")
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            name, data = next(iter(uploaded.items()))
            webcam_path.write_bytes(data)
            postprocess_install_framing(webcam_path)
            print(f"Saved upload '{name}' to {webcam_path}")
            display(Image(str(webcam_path)))
else:
    from google.colab import files
    print("Choose a photo to use as webcam input.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file uploaded.")
    name, data = next(iter(uploaded.items()))
    webcam_path.write_bytes(data)
    postprocess_install_framing(webcam_path)
    print(f"Saved upload '{name}' to {webcam_path}")
    display(Image(str(webcam_path)))

In [ ]:
#@title webcam 2 prompt
# Run the **capture webcam** cell first (or enable installation_mode for continuous capture).

WEBCAM_FILENAME = "webcam.png"
installation_mode = False  #@param {type:"boolean"}  # True = loop forever (re-captures each round in Colab)
prompt_mode = 'fast' #@param ["best","fast","classic","negative"]
output_mode = 'desc.csv'

height = 600 #@param {type:"integer"}
width = 800 #@param {type:"integer"}
num_inference_steps = 30 #@param {type:"integer"}
guidance_scale = 7 #@param {type:"integer"}
n_images = 5 #@param {type:"integer"}


while True:
  folder_path = OUTPUT_DIR
  webcam_path = folder_path / WEBCAM_FILENAME

  if installation_mode and not webcam_path.exists():
      print("Installation mode: capturing new webcam frame…")
      capture_webcam_to_file(webcam_path)

  if not webcam_path.exists():
      print(f"No {WEBCAM_FILENAME} in {folder_path}. Run the **capture webcam** cell first.")
      if installation_mode:
          import time
          time.sleep(2)
          continue
      break

  ci.config.quiet = True
  files = [WEBCAM_FILENAME]
  prompts = []

  for idx, file in enumerate(tqdm(files, desc='Generating prompts')):
      image = Image.open(folder_path / file).convert('RGB')
      prompt = image_to_prompt(image, prompt_mode)
      prompts.append(prompt)
      print(prompt)
      thumb = image.copy()
      thumb.thumbnail([256, 256])
      display(thumb)

  if not prompts:
      print(f"Sorry, I couldn't find any images in {folder_path}")
      if installation_mode:
          continue
      break

  if output_mode == 'desc.csv':
      csv_path = folder_path / 'desc.csv'
      with open(csv_path, 'w', encoding='utf-8', newline='') as f:
          w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
          w.writerow(['image', 'prompt'])
          for file, prompt in zip(files, prompts):
              w.writerow([file, prompt])
      print(f"\n\nGenerated {len(prompts)} prompts and saved to {csv_path}")

      with open(PROMPT_FILE, 'w', encoding='utf-8') as f:
          f.write(prompts[0])
      print(prompts[0])

  for i in range(n_images):
    images = prompt_to_img(
        prompts[0], height=height, width=width,
        num_inference_steps=num_inference_steps, guidance_scale=guidance_scale,
        return_all_latents=True, folder=i)

  if installation_mode:
      # Live installation: capture again on next loop iteration
      webcam_path.unlink(missing_ok=True)
      continue

  print("Done. Generated images are in OUTPUT_DIR/<0..n>/")
  break